In [ ]:
# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ================================================================
# 1. LOAD DATA AND COLUMN MAPPING
# ================================================================
file_path = "Table1.csv"
df = pd.read_csv(file_path, low_memory=False)

col_gender_identity = next(
    (c for c in df.columns if "identidade de gênero" in c.lower()),
    "Identidade de gênero",
)
col_sex = next(
    (c for c in df.columns if "sexo" in c.lower()),
    "Sexo",
)

infectious_cols = [
    c for c in df.columns
    if "doenças infecciosas" in c.lower()
    and "choice=" in c.lower()
    and "não" not in c.lower()
]

chronic_cols = [
    c for c in df.columns
    if "doenças crônicas" in c.lower()
    and "choice=" in c.lower()
    and "não" not in c.lower()
]

specialty_cols = [
    c for c in df.columns
    if ("encaminhamento" in c.lower() or "especialidade" in c.lower())
    and "choice=" in c.lower()
]

ciap_cols = [c for c in df.columns if "ciap" in c.lower()]

# ================================================================
# 2. TRANSLATION MAPS
# ================================================================
specialty_translation = {
    "Cirurgia Geral": "General Surgery",
    "Endocrinologia": "Endocrinology",
    "Infectologia": "Infectious Diseases",
    "Oftalmologia": "Ophthalmology",
    "Ortopedia": "Orthopedics",
    "Psiquiatria": "Psychiatry",
    "Psicologia": "Psychology",
    "Cardiologia": "Cardiology",
    "Dermatologia": "Dermatology",
    "Neurologia": "Neurology",
    "Ginecologia": "Gynecology",
    "Urologia": "Urology",
    "Pneumologia": "Pulmonology",
    "Nefrologia": "Nephrology",
    "Hematologia": "Hematology",
    "Gastroenterologia": "Gastroenterology",
    "Reumatologia": "Rheumatology",
    "Otorrinolaringologia": "Otorhinolaryngology",
    "Mastologia": "Breast Surgery",
    "Oncologia": "Oncology",
    "Nutrição": "Nutrition",
    "Nutricionista": "Nutrition",
    "Serviço Social": "Social Work",
    "Fisioterapia": "Physiotherapy",
    "Fonoaudiologia": "Speech Therapy",
    "Saúde Mental": "Mental Health",
}

def translate_specialty(value):
    value_str = str(value).strip()
    return specialty_translation.get(value_str, value_str)

# ================================================================
# 3. LONGITUDINAL TREATMENT AND GENDER IMPUTATION
# ================================================================
df = df.sort_values(["Record ID", "Repeat Instance"], na_position="first")

cols_to_fill = [col_gender_identity, col_sex] + infectious_cols + chronic_cols + specialty_cols
cols_to_fill = [c for c in cols_to_fill if c and c in df.columns]

df[cols_to_fill] = df.groupby("Record ID")[cols_to_fill].ffill()
df_visits = df[df["Repeat Instrument"].notna()].copy()

def smart_gender_mapping(row):
    gender_identity = str(row.get(col_gender_identity, "")).lower().strip()
    sex = str(row.get(col_sex, "")).lower().strip()

    if gender_identity == "nan" or gender_identity == "" or "prefere não" in gender_identity:
        if "masculino" in sex or "homem" in sex or sex == "m":
            return "Cis Man"
        if "feminino" in sex or "mulher" in sex or sex == "f":
            return "Cis Woman"
        return None

    if "homem transgênero" in gender_identity:
        return "Trans Man"
    if "mulher transgênero" in gender_identity or "travesti" in gender_identity:
        return "Trans Woman"
    if "homem cisgênero" in gender_identity:
        return "Cis Man"
    if "mulher cisgênero" in gender_identity:
        return "Cis Woman"

    return None

df_visits["Gender_Group"] = df_visits.apply(smart_gender_mapping, axis=1)
df_plot = df_visits.dropna(subset=["Gender_Group"]).copy()

df_plot["Total_Disease_Burden"] = (
    (df_plot[infectious_cols] == "Checked").sum(axis=1)
    + (df_plot[chronic_cols] == "Checked").sum(axis=1)
)

# ================================================================
# 4. MELTING FOR SPECIALTY ANALYSIS
# ================================================================
df_melted = df_plot.melt(
    id_vars=["Record ID", "Gender_Group", "Total_Disease_Burden"],
    value_vars=specialty_cols,
    var_name="Specialty",
    value_name="Checked_Status",
)

df_melted = df_melted[df_melted["Checked_Status"] == "Checked"].copy()
df_melted["Specialty"] = df_melted["Specialty"].apply(
    lambda x: x.split("=")[-1].replace(")", "").strip()
)
df_melted["Specialty_EN"] = df_melted["Specialty"].apply(translate_specialty)

target_specialties = set()

for group in df_melted["Gender_Group"].unique():
    top_group_specialties = (
        df_melted[df_melted["Gender_Group"] == group]["Specialty_EN"]
        .value_counts()
        .head(3)
        .index
    )
    target_specialties.update(top_group_specialties)

for specialty in df_melted["Specialty_EN"].unique():
    specialty_lower = specialty.lower()
    if (
        "psychiatry" in specialty_lower
        or "psychology" in specialty_lower
        or "mental health" in specialty_lower
    ):
        target_specialties.add(specialty)

df_final = df_melted[df_melted["Specialty_EN"].isin(target_specialties)].copy()
specialty_order = sorted(list(target_specialties))

# ================================================================
# 5. EXPORT ARTICLE TABLES
# ================================================================
complexity_table = (
    df_final.groupby(["Specialty_EN", "Gender_Group"])["Total_Disease_Burden"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

complexity_table.columns = [
    "Medical Specialty",
    "Gender Identity",
    "Mean Disease Burden",
    "Standard Deviation",
    "N (Visits)",
]

complexity_table_file = "clinical_complexity_by_gender_en.csv"
complexity_table.to_csv(complexity_table_file, index=False, encoding="utf-8-sig")

ciap_rows = []

if ciap_cols:
    target_ciap_col = ciap_cols[0]

    for group in ["Cis Woman", "Cis Man", "Trans Woman", "Trans Man"]:
        subgroup = df_plot[df_plot["Gender_Group"] == group]
        top_ciaps = subgroup[target_ciap_col].dropna().value_counts().head(5)

        for ciap, count in top_ciaps.items():
            ciap_rows.append(
                {
                    "Gender Identity": group,
                    "Total N (Group)": len(subgroup),
                    "CIAP-2 Reason": str(ciap).strip(),
                    "Frequency": count,
                    "Proportion (%)": round((count / len(subgroup) * 100), 1) if len(subgroup) > 0 else 0,
                }
            )

    ciap_table_file = "ciap_by_gender_en.csv"
    pd.DataFrame(ciap_rows).to_csv(ciap_table_file, index=False, encoding="utf-8-sig")
else:
    ciap_table_file = None

# ================================================================
# 6. CONSOLE REPORT — CIAP-2 BY GENDER
# ================================================================
print(f"\n{'=' * 90}")
print("MAIN REASONS FOR CONSULTATION (CIAP-2) BY GENDER IDENTITY")
print(f"{'=' * 90}")

if not ciap_cols:
    print("No column containing the term 'CIAP' was found.")
else:
    target_ciap_col = ciap_cols[0]
    groups = ["Cis Woman", "Cis Man", "Trans Woman", "Trans Man"]

    for group in groups:
        subgroup = df_plot[df_plot["Gender_Group"] == group]
        print(f"\n[{group.upper()}] - Total visits in the sample (after imputation): {len(subgroup)}")

        if len(subgroup) > 0:
            top_ciaps = subgroup[target_ciap_col].dropna().value_counts().head(3)
            if len(top_ciaps) > 0:
                for ciap, count in top_ciaps.items():
                    print(f"  -> {str(ciap)[:60]:<60} ({count} cases)")
            else:
                print("  -> No CIAP registered for patients in this group.")

print(f"\n{'=' * 90}")

# ================================================================
# 7. FIGURE — BARPLOT
# ================================================================
plt.figure(figsize=(14, 10))
sns.set_theme(style="whitegrid")

palette = {
    "Cis Woman": "#95a5a6",
    "Cis Man": "#34495e",
    "Trans Woman": "#9b59b6",
    "Trans Man": "#e67e22",
}

ax = sns.barplot(
    data=df_final,
    y="Specialty_EN",
    x="Total_Disease_Burden",
    hue="Gender_Group",
    palette=palette,
    capsize=0.1,
    err_kws={"linewidth": 1.5},
    order=specialty_order,
    hue_order=["Cis Woman", "Cis Man", "Trans Woman", "Trans Man"],
)

plt.title(
    "Clinical Complexity (Mean Disease Burden) by Specialty and Gender",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
plt.xlabel(
    "Mean Number of Detected Diseases per Patient (Infectious + Chronic)",
    fontsize=13,
    fontweight="bold",
)
plt.ylabel(
    "Requested Medical Specialty",
    fontsize=13,
    fontweight="bold",
)

sns.despine(left=True, bottom=True)
plt.legend(
    title="Gender Identity",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=False,
    fontsize=11,
)
plt.tight_layout()
plt.show()

# ================================================================
# 8. FINAL EXPORT MESSAGE
# ================================================================
print("\nExported files:")
print(f" - '{complexity_table_file}'")
if ciap_table_file:
    print(f" - '{ciap_table_file}'")